# ViGoEmotions — Kaggle: phobert tuning
Attach hai Dataset riêng tư: code và ba CSV. Bật GPU + Internet.
Chạy từng cell từ trên xuống; nếu lỗi thì dừng. Hướng dẫn: docs/kaggle_guide.md.
Cell đầu chỉ chạy một lần trong session mới. Kết quả lưu trong /kaggle/working.
Đây là một lượt chạy mới; không dùng để resume sweep Colab khác môi trường.


In [ ]:
from pathlib import Path
import os, shutil, zipfile

INPUT = Path('/kaggle/input')
ROOT = Path('/kaggle/working/ViGoEmotions')
if ROOT.exists():
    raise FileExistsError(f'{ROOT} đã tồn tại. Không giải nén đè lên kết quả; chạy tiếp cell cần thiết hoặc dùng session mới.')

# Kaggle có thể giữ ZIP hoặc tự giải nén ZIP khi tạo Dataset.
archives = list(INPUT.rglob('vigoemotions_colab.zip'))
code_roots = [p.parent.parent for p in INPUT.rglob('scripts/common.py')]
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        for item in archive.infolist():
            if not (ROOT / item.filename).resolve().is_relative_to(ROOT.resolve()):
                raise ValueError('Unsafe ZIP path')
        ROOT.mkdir(parents=True)
        archive.extractall(ROOT)
elif not archives and len(code_roots) == 1:
    shutil.copytree(code_roots[0], ROOT)
else:
    raise RuntimeError('Hãy attach đúng một Dataset code chứa vigoemotions_colab.zip hoặc project đã giải nén.')
os.chdir(ROOT)
print('Project:', ROOT)


In [ ]:
%pip install -q -r requirements-colab.txt
import shutil, subprocess
if shutil.which('java') is None:
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'default-jre-headless'], check=True)


In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), 'Hãy bật GPU runtime trước khi chạy tiếp.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['java','-version'], check=True)

## Nạp dữ liệu
Attach Dataset chứa train.csv, val.csv, test.csv vào Input. Cell dưới copy CSV vào project để các script dùng chung đường dẫn.


In [ ]:
from pathlib import Path
import csv, shutil

sources = {}
for name in ('train.csv', 'val.csv', 'test.csv'):
    candidates = list(Path('/kaggle/input').rglob(name))
    if len(candidates) != 1:
        raise RuntimeError(f'Cần đúng một {name} trong Input; tìm thấy {len(candidates)}. Kiểm tra Dataset đã attach.')
    with candidates[0].open(encoding='utf-8-sig', newline='') as handle:
        if next(csv.reader(handle)) != ['id', 'text', 'labels']:
            raise ValueError(f'{name}: schema phải là id,text,labels')
    sources[name] = candidates[0]
(ROOT / 'data').mkdir(exist_ok=True)
for name, source in sources.items():
    shutil.copy2(source, ROOT / 'data' / name)
    print('Đã nạp:', name)


## Kiểm tra gói code và baseline
Cần ZIP có `--include-results` để report giữ được đủ baseline và EDA. Nếu đã chạy sweep trước đó, dùng runtime/gói mới với thư mục sweep chưa có kết quả.

In [ ]:
import sys, subprocess, json
sys.path.insert(0, str(ROOT/'scripts'))
for name in ('tfidf_lr','gru','phobert'):
    assert Path(f'reports/tables/{name}_metrics.csv').exists(), 'Tạo lại ZIP với --include-results'
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],check=True)
import torch
assert torch.cuda.is_available(), 'Hãy bật GPU runtime'
print('GPU:', torch.cuda.get_device_name(0))

## Chạy ba learning rate
Cả ba bắt đầu từ pretrained revision gốc. Script lưu từng lượt vào thư mục riêng và kiểm tra kết quả từ xác suất. Thư mục kết quả/checkpoint đã tồn tại sẽ bị từ chối để tránh ghi đè. Thời gian baseline cũ khoảng 51 phút/lượt; thời gian thực tế sweep chưa đo.

In [ ]:
import py_vncorenlp
segmenter_dir = (ROOT/'data/vncorenlp').resolve()
segmenter_dir.mkdir(parents=True,exist_ok=True)
py_vncorenlp.download_model(save_dir=str(segmenter_dir))
subprocess.run([sys.executable,'scripts/tune_phobert.py','run',
                '--vncorenlp-dir',str(segmenter_dir)],check=True)

## Xem kết quả và tạo lại report/slide
Các insight về learning rate cần dựa trên số liệu mới. Bảng ca lỗi tự sinh còn cần nhóm đọc thủ công; không đưa comment nguồn vào báo cáo công khai.

In [ ]:
import pandas as pd
from IPython.display import display, Image
subprocess.run([sys.executable,'scripts/build_report.py'],check=True)
display(pd.read_csv('reports/phobert_tuning/comparison.csv'))
display(pd.read_csv('reports/phobert_tuning/styles_comparison.csv'))
display(Image(filename='reports/phobert_tuning/learning_curves.png'))
display(pd.read_csv('reports/phobert_tuning/neural_error_review.csv').head())

In [ ]:
with zipfile.ZipFile('/kaggle/working/vigoemotions_tuning_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for folder in ('reports','splits'):
        for p in Path(folder).rglob('*'):
            if p.is_file():
                z.write(p,str(p))
print('Output cần tải:', '/kaggle/working/vigoemotions_tuning_results.zip')


## Tải checkpoint được chọn để giữ cho Final
Lưu file này riêng tư trước khi runtime hết hạn. Checkpoint có kích thước lớn và không nằm trong ZIP kết quả. Chưa đánh giá test; Final sẽ dùng checkpoint đã chốt để so sánh ba cách chọn threshold.

In [ ]:
study=json.loads(Path('reports/phobert_tuning/study.json').read_text())
selected=study['selected_run']
run=json.loads(Path(f'reports/phobert_tuning/{selected}/phobert_run.json').read_text())
checkpoint=Path(run['checkpoint_path'])
assert checkpoint.is_file()
print('Output cần tải:', str(checkpoint))
